In [113]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [114]:
from typing import TypedDict
import yfinance as yf
from newsapi import NewsApiClient

In [115]:
# import dependencies
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

In [116]:
# Create an instance of the ChatGroq class, which will be used to interact with the Groq API for chatbot functionality
chat_groq = ChatGroq(model="qwen/qwen3-32b", temperature=0.7,max_tokens=500)

In [117]:
MARKET_ANALYSIS_PROMPT = """
You are a Senior Equity Research Analyst.
Stock: {ticker}
Market Data:
{market_data}
Analyze:
1. Trend
2. Volatility
3. Support/Resistance
4. Risk factors
Provide concise insights.
"""
NEWS_ANALYSIS_PROMPT = """
You are a Financial News Analyst.
News Articles:
{news}
Tasks:
1. Summarize events.
2. Identify sentiment.
3. Detect market impact.
Return structured output.
"""
RECOMMENDATION_PROMPT = """
You are a Portfolio Manager.
Market Analysis:
{analysis}
News Insights:
{news_insights}
Generate:
Recommendation:
(BUY, HOLD, SELL)
Confidence Score: %
Justification:
- Fundamental factors
- Technical factors
- News sentiment
Risk Warning:
"""

In [118]:
from typing import TypedDict, List, Dict, Any

class StockState(TypedDict):
    ticker: str
    market_data: dict
    news_articles: list
    retrieved_context: str
    market_analysis: str
    news_analysis: str
    recommendation: str

In [119]:
#Agent 1: Stock Data Agent

def market_agent(state):
    ticker = state["ticker"]
    stock = yf.Ticker(ticker)
    hist = stock.history(period="6mo")
    latest_price = hist["Close"].iloc[-1]
    sma20 = hist["Close"].rolling(20).mean().iloc[-1]
    sma50 = hist["Close"].rolling(50).mean().iloc[-1]
    market_data = {
        "latest_price": float(latest_price),
        "sma20": float(sma20),
        "sma50": float(sma50),
        "volume": float(hist["Volume"].iloc[-1])
    }
    state["market_data"] = market_data
    return state

In [120]:
# NEWS AGENT.


newsapi = NewsApiClient(
    api_key=os.getenv("NEWS_API_KEY")
)

def news_agent(state):

    ticker = state["ticker"]
    articles = newsapi.get_everything(
        q=ticker,
        language="en",
        sort_by="publishedAt",
        page_size=10
    )
    news_docs = []
    for article in articles["articles"]:
        text = f"""
        Title: {article['title']}
        Description: {article['description']}
        """
        news_docs.append(text)
    state["news_articles"] = news_docs
    return state

In [121]:
#pip install chromadb
#pip install langchain-openai

In [122]:
# RAG AGENT.
import chromadb
from langchain_openai import OpenAIEmbeddings
client = chromadb.PersistentClient(path="./vectorstore")
collection = client.get_or_create_collection(name="financial_news")
embeddings = OpenAIEmbeddings()
def rag_agent(state):
    docs = state["news_articles"]
    for idx, doc in enumerate(docs):
        collection.add(
            ids=[str(idx)],
            documents=[doc]
        )
    query = f"{state['ticker']} market sentiment"

    results = collection.query(query_texts=[query],n_results=5)
    retrieved_context = "\n".join(
        results["documents"][0]
    )
    state["retrieved_context"] = retrieved_context
    return state

In [123]:
# Market Analysis Agent
#from prompts.templates import MARKET_ANALYSIS_PROMPT

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0)
def analyst_agent(state):
    prompt = MARKET_ANALYSIS_PROMPT.format(
        ticker=state["ticker"],
        market_data=state["market_data"]
    )
    response = llm.invoke(prompt)
    state["market_analysis"] = response.content
    return state

In [124]:
# recommendation agent.
#from prompts.templates import RECOMMENDATION_PROMPT
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0
)
def recommendation_agent(state):
    prompt = RECOMMENDATION_PROMPT.format(
        analysis=state["market_analysis"],
        news_insights=state["retrieved_context"]
    )
    response = llm.invoke(prompt)
    state["recommendation"] = response.content
    return state

In [125]:
from langgraph.graph import StateGraph

builder = StateGraph(StockState)

builder.add_node("market_agent",market_agent)

builder.add_node("news_agent",news_agent)

builder.add_node("rag_agent",rag_agent)

builder.add_node("analyst_agent",analyst_agent)

builder.add_node("recommendation_agent",recommendation_agent)

builder.set_entry_point("market_agent")

builder.add_edge("market_agent","news_agent")

builder.add_edge("news_agent","rag_agent")

builder.add_edge("rag_agent","analyst_agent")

builder.add_edge("analyst_agent","recommendation_agent")

builder.set_finish_point("recommendation_agent")

graph = builder.compile()

In [126]:

if __name__ == "__main__":

    ticker = input("Enter stock ticker: ")

    result = graph.invoke(
        {
            "ticker": ticker
        }
    )

    print(
        "\n===== FINAL REPORT =====\n"
    )

    print(
        result["recommendation"]
    )


===== FINAL REPORT =====

**Recommendation:**
BUY

**Confidence Score:**
75%

**Justification:**

*   **Fundamental Factors:**
    *   **Analyst Support:** JP Morgan recently raised its price target to $360, signaling institutional confidence despite recent noise regarding iPhone 18 Pro production cuts. This suggests that the market is overreacting to short-term supply chain headlines rather than fundamental demand issues.
    *   **Market Position:** Apple remains a core holding in the "Magnificent 7" tech cohort, having reclaimed its position among the $4 trillion market cap club alongside Microsoft, Nvidia, and Google. This status provides a floor of institutional demand.
    *   **Long-term Structural View:** While BofA warns of long-term risks from Starlink’s spectrum acquisition, this is a multi-year horizon issue that does not impact immediate trading dynamics or current earnings power.

*   **Technical Factors:**
    *   **Trend Confirmation:** The stock is in a clear short-to